# Variance-Dependent Time Series Model

This notebook implements a time series model where variance depends on price movements:

$$\text{Var} = \sigma^2 + \frac{z^2}{2}$$

where:
- $z = \frac{x}{\sqrt{T}}$
- $x$ is the drift-corrected log price change
- $\sigma$ is the base volatility
- $T$ is the time horizon

## Setup and Load Libraries

In [ ]:
# Install packages if needed (uncomment to run)
# install.packages(c("ggplot2", "gridExtra"))

library(ggplot2)
library(gridExtra)

# Set plotting options
options(repr.plot.width = 12, repr.plot.height = 8)

: 

## Define Model Parameters

In [ ]:
# Model parameters
n_periods <- 252    # Number of time periods
T <- 1.0            # Time horizon
sigma <- 0.2        # Base volatility
mu <- 0.05          # Drift (average return)
seed <- 42          # Random seed for reproducibility

cat("Model Parameters:\n")
cat(sprintf("  Base Volatility (σ): %.2f\n", sigma))
cat(sprintf("  Drift (μ): %.2f\n", mu))
cat(sprintf("  Time Horizon (T): %.2f\n", T))
cat(sprintf("  Number of Periods: %d\n", n_periods))
cat(sprintf("  Random Seed: %d\n", seed))

## Core Function: Generate Time Series

In [ ]:
generate_variance_dependent_series <- function(n_periods = 252, T = 1.0, sigma = 0.2, mu = 0.05, seed = 42) {
  """
  Generate a time series where variance over period T satisfies:
  Var = sigma^2 + z^2/2
  where z = x/sqrt(T) and x is the log price change (drift-corrected)
  
  Parameters:
  -----------
  n_periods : int
      Number of time periods to simulate
  T : float
      The time horizon for variance calculation
  sigma : float
      Base volatility parameter
  mu : float
      Drift (average return)
  seed : int
      Random seed for reproducibility
  
  Returns:
  --------
  A list containing:
    - prices: price path
    - log_prices: log price path
    - z_values: standardized drift-corrected moves
    - theoretical_vars: variance at each time step
  """
  
  set.seed(seed)
  
  dt <- T / n_periods
  
  # Initialize arrays
  log_prices <- numeric(n_periods + 1)
  prices <- numeric(n_periods + 1)
  prices[1] <- 100.0
  log_prices[1] <- log(prices[1])
  
  # Storage for analysis
  z_values <- numeric(n_periods)
  theoretical_vars <- numeric(n_periods)
  
  for (i in 1:n_periods) {
    # Calculate x (drift-corrected log price change)
    if (i == 1) {
      x <- 0
    } else {
      period_length <- min((i - 1) * dt, T)
      x <- log_prices[i] - log_prices[1] - mu * period_length
    }
    
    # Calculate z
    period_T <- min(i * dt, T)
    z <- if (period_T > 0) x / sqrt(period_T) else 0
    
    # Calculate variance for this step
    var_t <- sigma^2 + z^2 / 2
    std_t <- sqrt(var_t)
    
    # Generate return with this variance
    dW <- rnorm(1, mean = 0, sd = sqrt(dt))
    dlog_price <- mu * dt + std_t * dW
    
    log_prices[i + 1] <- log_prices[i] + dlog_price
    prices[i + 1] <- exp(log_prices[i + 1])
    
    # Store for analysis
    z_values[i] <- z
    theoretical_vars[i] <- var_t
  }
  
  list(
    prices = prices,
    log_prices = log_prices,
    z_values = z_values,
    theoretical_vars = theoretical_vars
  )
}

## Generate the Time Series

In [ ]:
# Generate the series
cat("Generating time series...\n")
result <- generate_variance_dependent_series(
  n_periods = n_periods,
  T = T,
  sigma = sigma,
  mu = mu,
  seed = seed
)

cat("Generation complete!\n")

## Summary Statistics

In [ ]:
# Calculate statistics
initial_price <- result$prices[1]
final_price <- result$prices[length(result$prices)]
total_return <- (final_price / initial_price - 1) * 100
log_return <- result$log_prices[length(result$log_prices)] - result$log_prices[1]

cat("\n=== SUMMARY STATISTICS ===\n\n")

cat("Price Statistics:\n")
cat(sprintf("  Initial price: %.2f\n", initial_price))
cat(sprintf("  Final price: %.2f\n", final_price))
cat(sprintf("  Total return: %.2f%%\n", total_return))
cat(sprintf("  Log return: %.4f\n", log_return))

cat("\nVariance Statistics:\n")
cat(sprintf("  Base variance (σ²): %.4f\n", sigma^2))
cat(sprintf("  Mean realized variance: %.4f\n", mean(result$theoretical_vars)))
cat(sprintf("  Min variance: %.4f\n", min(result$theoretical_vars)))
cat(sprintf("  Max variance: %.4f\n", max(result$theoretical_vars)))

cat("\nZ Statistics:\n")
cat(sprintf("  Mean z value: %.4f\n", mean(result$z_values)))
cat(sprintf("  Std of z values: %.4f\n", sd(result$z_values)))
cat(sprintf("  Min z: %.4f\n", min(result$z_values)))
cat(sprintf("  Max z: %.4f\n", max(result$z_values)))

## Visualization 1: Price Paths

In [ ]:
# Prepare data for plotting
time_full <- seq(0, T, length.out = length(result$prices))
time_short <- seq(T / n_periods, T, length.out = n_periods)

# Price plot
df_price <- data.frame(time = time_full, price = result$prices)
p1 <- ggplot(df_price, aes(x = time, y = price)) +
  geom_line(color = "darkblue", linewidth = 1.2) +
  labs(title = "Simulated Price Path", x = "Time", y = "Price") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15))

# Log price plot
df_logprice <- data.frame(time = time_full, log_price = result$log_prices)
p2 <- ggplot(df_logprice, aes(x = time, y = log_price)) +
  geom_line(color = "darkgreen", linewidth = 1.2) +
  labs(title = "Log Price Path", x = "Time", y = "Log Price") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15))

# Display plots
grid.arrange(p1, p2, ncol = 1)

## Visualization 2: Variance Analysis

In [ ]:
# Z values plot
df_z <- data.frame(time = time_short, z = result$z_values)
p3 <- ggplot(df_z, aes(x = time, y = z)) +
  geom_line(color = "darkred", linewidth = 1.2) +
  geom_hline(yintercept = 0, linetype = "dashed", alpha = 0.5, linewidth = 0.8) +
  labs(title = "Standardized Drift-Corrected Move (z)", 
       x = "Time", y = "z = x/√T") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15))

# Variance plot
df_var <- data.frame(time = time_short, variance = result$theoretical_vars)
p4 <- ggplot(df_var, aes(x = time, y = variance)) +
  geom_line(color = "purple", linewidth = 1.3) +
  geom_hline(aes(yintercept = sigma^2), 
             color = "orange", linetype = "dashed", linewidth = 1.3) +
  annotate("text", x = T * 0.8, y = sigma^2 * 1.1, 
           label = paste0("σ² = ", round(sigma^2, 4)), 
           color = "orange", size = 5, fontface = "bold") +
  labs(title = "Variance: σ² + z²/2", x = "Time", y = "Variance") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15))

# Display plots
grid.arrange(p3, p4, ncol = 1)

## Visualization 3: Variance Decomposition

In [ ]:
# Scatter plot: z² vs variance component
z_squared <- result$z_values^2
var_component <- z_squared / 2
df_scatter <- data.frame(z_squared = z_squared, var_component = var_component)

p5 <- ggplot(df_scatter, aes(x = z_squared, y = var_component)) +
  geom_point(alpha = 0.5, size = 2.5, color = "darkblue") +
  geom_abline(slope = 0.5, intercept = 0, color = "red", 
              linetype = "dashed", linewidth = 1.3) +
  annotate("text", x = max(z_squared) * 0.7, y = max(var_component) * 0.9, 
           label = "y = z²/2", color = "red", size = 5, fontface = "bold") +
  labs(title = "Variance Component vs z²", 
       x = "z²", y = "z²/2 component") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15))

# Decomposition plot
base_var <- rep(sigma^2, n_periods)
df_decomp <- data.frame(
  time = time_short,
  base = base_var,
  total = result$theoretical_vars,
  additional = result$theoretical_vars - base_var
)

p6 <- ggplot(df_decomp) +
  geom_ribbon(aes(x = time, ymin = 0, ymax = base, fill = "Base: σ²"), alpha = 0.6) +
  geom_ribbon(aes(x = time, ymin = base, ymax = total, fill = "Additional: z²/2"), alpha = 0.6) +
  geom_line(aes(x = time, y = total), color = "black", linewidth = 1.3) +
  scale_fill_manual(values = c("Base: σ²" = "steelblue", "Additional: z²/2" = "coral"),
                    name = "Variance Component") +
  labs(title = "Variance Decomposition", x = "Time", y = "Variance") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15),
        legend.position = "top",
        legend.title = element_text(face = "bold"))

# Display plots
grid.arrange(p5, p6, ncol = 1)

## Data Export

In [ ]:
# Create a data frame with all results
df_export <- data.frame(
  time = time_full,
  price = result$prices,
  log_price = result$log_prices,
  z_value = c(NA, result$z_values),
  variance = c(NA, result$theoretical_vars)
)

# Display first few rows
cat("\nFirst 10 rows of data:\n")
print(head(df_export, 10))

# Optionally save to CSV (uncomment to save)
# write.csv(df_export, "variance_timeseries_data.csv", row.names = FALSE)
# cat("\nData saved to variance_timeseries_data.csv\n")

## Experiment: Different Parameter Values

Try modifying the parameters in the cell below to see how the model behaves differently:

In [ ]:
# Experiment with different parameters
cat("\n=== EXPERIMENT: HIGH VOLATILITY ===\n")

result_high_vol <- generate_variance_dependent_series(
  n_periods = 252,
  T = 1.0,
  sigma = 0.4,  # Higher base volatility
  mu = 0.05,
  seed = 42
)

# Quick comparison plot
df_compare <- data.frame(
  time = time_full,
  original = result$prices,
  high_vol = result_high_vol$prices
)

ggplot(df_compare) +
  geom_line(aes(x = time, y = original, color = "σ = 0.2"), linewidth = 1.2) +
  geom_line(aes(x = time, y = high_vol, color = "σ = 0.4"), linewidth = 1.2) +
  scale_color_manual(values = c("σ = 0.2" = "darkblue", "σ = 0.4" = "darkred"),
                     name = "Volatility") +
  labs(title = "Comparison: Original vs High Volatility", 
       x = "Time", y = "Price") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 15),
        legend.position = "top")

## Model Interpretation

### Key Insights:

1. **Base Variance**: The model always has a minimum variance of $\sigma^2$, representing the baseline uncertainty in the process.

2. **Path-Dependent Volatility**: When the price deviates significantly from its expected drift path, the variance increases through the $z^2/2$ term.

3. **Volatility Clustering**: Large moves tend to be followed by higher volatility, creating realistic clustering effects.

4. **Symmetric Response**: The variance responds to the magnitude of deviations (through $z^2$), not their direction.

5. **Time Scaling**: The standardization by $\sqrt{T}$ ensures that variance scales appropriately with the time horizon.

### Applications:
- Modeling financial assets with volatility feedback
- Risk management with path-dependent uncertainty
- Scenario analysis for extreme market conditions